In [42]:
from uuid import uuid4
from langchain_community.document_loaders import PyPDFLoader
from dotenv import load_dotenv
import os
load_dotenv()
#os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
google_api_key = os.getenv("OPENAI_API_KEY")

if not google_api_key:
    raise ValueError("OPENAI_API_KEY is missing")

# uv pip install langchain_qdrant

In [43]:
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

In [44]:
COLLECTION_NAME = "company-policy-rag"

In [45]:
loader = PyPDFLoader("E:\workspace\Database\chromadb\data\llama2-research-paper.pdf")
pages = loader.load()
print("PDF pages loaded:", len(pages))

Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


PDF pages loaded: 77


In [46]:
# Split docs
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=120,
)

chunks=text_splitter.split_documents(pages)
print("chunks created:", len(chunks))

chunks created: 174


In [47]:
# Add useful metedata

for chunk_index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = chunk_index
    chunk.metadata["filename"] = "llama2-research-paper.pdf"

In [49]:
# Embedding
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings(model="text-embedding-3-large")
try:
    vector = embeddings.embed_query("dimension check")

    print("Type:", type(vector))
    print("Dimension:", len(vector))

except Exception as e:
    print("Error type:", type(e).__name__)
    print("Error message:", str(e))



Error type: RateLimitError
Error message: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}


In [50]:
import os
from dotenv import load_dotenv
load_dotenv()
qdrant_key = os.getenv("QDRANT_KEY")
qdrant_endpoint = os.getenv("QDRANT_ENDPOINT_CLIENT")

In [51]:
# Client to access
client = QdrantClient(api_key=qdrant_key,url=qdrant_endpoint)

In [ ]:
# Create a Collection
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=dimension,
            distance=models.Distance.COSINE
        ),
    )


In [ ]:
# Qdrant Vector store
if chunks and isinstance(chunks[0], str):
    documents = [
        Document(page_content=text)
        for text in chunks
    ]
else:
    documents = chunks,
    vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings
)

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [ ]:
# Use stable id's in production
chunk_ids = [
    str(uuid4())
    for _ in chunks
]

In [ ]:
insert_ids = await vector_store.aadd_documents(
    documents=documents,
    ids=chunk_ids
)

print("Inserted IDs:", insert_ids)

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_tokens, limit: 30000, model: gemini-embedding-1.0, description: Embed content tokens per minute limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 40.047629479s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Embed content tok